<a href="https://colab.research.google.com/github/RazzberryBoy26/DP-HE-FL-DaSHLab-/blob/main/Attention_PosEnc_Transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Importing all needed libraries.
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter
from datasets import load_dataset
from transformers import GPT2Tokenizer

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

###########################################
# 1. Loading and processing the dataset.
###########################################

print("Loading official OpenAI GPT-2 BPE Tokenizer...")
# We call the tokenizer from our GPT2Tokenizer class.
# This downloads the exact 50,257-subword dictionary and merge rules used by GPT-2.
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

vocab_size = tokenizer.vocab_size
print(f"GPT-2 Vocabulary Size Verified: {vocab_size}")

dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")

# This tokenizer functions on BPE (Byte - Pair encoding).
# The token split is generated on basis of certain common occuring sets of letters, instead of separate words now.
# eg: "unambiguously" = "un" + "ambiguous" + "ly". This helps for better word pairing whenever needed.
# Earlier we were using Regex tokenizer- Tokens are hard separated with spacings. This is non-optimal for contextual learning.
# Now, we process the dataset - conversion into a tensor.
def data_process(raw_text_dataset):
    data = []
    for line in raw_text_dataset["text"]:
        # Skip purely empty paragraph spacing lines to avoid training on blank noise.
        if len(line.strip()) == 0:
            continue
        # BPE converts strings to compressed subword ID integers natively.
        # Natively handles unknown words by breaking them down into known byte-level chunks.
        tokenized_line = tokenizer.encode(line)
        data.extend(tokenized_line)

    # Return a 1D tensor containing the entire dataset sequence consecutively.
    return torch.tensor(data, dtype = torch.long)

# After mapping the vocabulary to different ID's, we set up the training and validation tensors.
print("Processing data splits into BPE subwords...")
train_tensor = data_process(dataset["train"])
val_tensor = data_process(dataset["validation"])

# GPT-2 standard baseline alignment batching
def batchify(data, batch_size):
    # Calculates how many clean batches we can fit.
    nbatch = data.size(0) // batch_size
    # Trims off any leftover tokens at the end that don't cleanly fit into the batch size.
    data = data.narrow(0, 0, nbatch * batch_size)
    # Reshapes the 1D tensor into 2D: [Batch_Size, Total_Tokens_Per_Batch]
    # Then transposes (.t()) it so that the shape becomes [Tokens, Batch_Size].
    # This allows sequence chunks to flow continuously down the columns over time.
    data = data.view(batch_size, -1).t().contiguous()
    return data

# Reduced to 8 to fit d_model=512 into 15GB VRAM
batch_size = 8
train_data = batchify(train_tensor, batch_size)
val_data = batchify(val_tensor, batch_size)

def get_batch(source, i, win_len = 1024):
    # Ensures we don't grab a window larger than the remaining text sequence.
    seq_len = min(win_len, len(source) - 1 - i)

    # Slices a chunk of input data of size [win_len, batch_size]
    data = source[i : i + seq_len]

    # Slices the chronological targets directly next to the inputs
    # Target is shifted exactly 1 token into the future (causal language modeling).
    target = source[i + 1 : i + 1 + seq_len]
    return data, target

Loading official OpenAI GPT-2 BPE Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


GPT-2 Vocabulary Size Verified: 50257
Processing data splits into BPE subwords...


In [ ]:
######################################################################
# 2.1.1 Positional Encoders: Sine-Cosine Absolute-Positional encoding
######################################################################

# The positional encoder used in the original Attention paper.
# Sets up our baseline model.
class PositionalEncoder(nn.Module):
    # Initializing needed tensors.
    def __init__(self, d_model, max_len = 5000):
        super().__init__()
        # Creates a matrix of 0's of dim [5000, d_model].
        # This is to store the actual positional encodings which we will add to our batch / sequence.
        pe = torch.zeros(max_len, d_model)
        # Makes a column vector of iterations from 0 to 4999.
        # We will use this to form our internal sine / cosine argument.
        position = torch.arange(0, max_len, dtype = torch.float).unsqueeze(1)
        # This will create a 1-dim tensor of the arguments that we will multiply with position column vector.
        # Then we put inside the sine and cosine functions as arguments to form our encoding tables.
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        # Add another index at that first axis for efficiency in further processing.
        pe = pe.unsqueeze(0)
        # We don't want to pe to get modified. Hence we store it in our GPU's registers as buffer.
        # Tell pyTorch this is NOT a learnable tensor.
        self.register_buffer('pe', pe)

    # Combining the positional encoder with dataset.
    # Since we added one more index at first axis, this will allow pyTorch to broadcast pe into;
    # The dataset's shape. This signifies compatability.
    def forward(self, x, current_pos = 0):
        new_seq_len = x.size(1)
        # We add the encoding table to all batches.
        x = x + self.pe[:, current_pos : current_pos + new_seq_len, :]
        return x

#############################################################
# 2.1.2 ALiBi positional encoding (Linear - Bias attention)
#############################################################

# This is a special type of encoding added directly to N - gram matrix.
# N - gram is nothing but the attention matrix formed per head.
# It adds a penalty to encodings that are too far away.
# For each head, we have a unique penalty factor, that is based off of a GP of 2.
class AlibiPositionalBias(nn.Module):
    # Initialize the tensors in advance if needed.
    def __init__(self, n_heads):
        super().__init__()
        self.n_heads = n_heads
        # This is to generate a tensor that contains the slopes for each head.
        # Each head will have a different slope, hence a different distance penalty for each token.
        slopes = self._get_slopes(n_heads)
        # These slopes are not learned, hence store them in the GPU register as buffer.
        self.register_buffer('slopes', torch.tensor(slopes).view(1, n_heads, 1, 1))

    # This function creates the slopes array, aka slope of each head.
    def _get_slopes(self, n_heads):
        # This is to get the slopes of first floor(log2(n)) heads.
        def get_slopes_power_of_2(n):
            # We define the standard formula for defining these slopes.
            start = (2 ** (-2 ** -(math.log2(n) - 3)))
            ratio = start
            # Creation of slopes 1D tensor.
            return [start * (ratio ** i) for i in range(n)]

        # If n_heads is a power of 2, then we want exactly log2(n) no. of slopes.
        if math.log2(n_heads).is_integer():
            return get_slopes_power_of_2(n_heads)
        # For thr other case, we first find out the 1D tensor corresponding to floor(log2(n_heads)).
        # Then recursively apply the same formula of slope finding on the next bunch of leftover heads.
        # This is almost the same as visualizing the no. of heads in binary format.
        else:
            # Fetch the first floor(log2(n_heads)) slopes.
            closest_power_of_2 = 2 ** math.floor(math.log2(n_heads))
            # Recursion on leftover heads.
            # Concatenation of the recursively formed 1D sub-tensor is done using '\' here.
            # This is nothing but a GP formation.
            return get_slopes_power_of_2(closest_power_of_2) + \
                   self._get_slopes(n_heads - closest_power_of_2)[:n_heads - closest_power_of_2]

    # Having found the values of slopes for each head stored in our tensor, we want to apply it to the attention matrix.
    def forward(self, seq_len, device):
        # Column and row vectors created of nos. from 0 to 1023.
        context_position = torch.arange(seq_len, device = device).unsqueeze(1)
        key_position = torch.arange(seq_len, device = device).unsqueeze(0)
        # Now, broadcasting allows us to calculate the relative distances between associated key and query entries.
        # In the N - gram matrix, we simply apply this to get our relative positional entries.
        relative_position = torch.abs(context_position - key_position)
        # We unsqueeze this with a few more dimensions such that we can add this directly into our N - gram matrix dataset.
        relative_position = relative_position.unsqueeze(0).unsqueeze(1)
        # Multiply this with slopes tensor in order to get our positional encodings.
        # These encodings are what we will directly add to attention matrix.
        alibi_bias = -1.0 * self.slopes * relative_position
        return alibi_bias

############################################
# 2.1.3 Rotary Positional Encoding (RoPE)
############################################

# Coding out the rotary positional encoding.
# This requires us to generate a rotation matrix.
# This transforms the key and query spaces respectively, such that it induces relative position info between all tokens.
class RotaryPositionalEmbedding(nn.Module):

    # We first initialize the rotation tensor for our keys and queries.
    def __init__(self, d_head, max_len = 5000, base = 10000):
        super().__init__()
        # This is to generate the values of frequencies we will put inside our sine and cosine arguments.
        # These sine and cosine functions will then form our rotation matrix.
        # This is different from absolute positional encodings, as we add a unit vector to each token.
        # Here, we rotate the axes themselves.
        inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
        # Store this in GPU registers as buffer tensor. We don't want to manipulate this.
        self.register_buffer("inv_freq", inv_freq)

        # Now, we generate a 1D array of positions of each token.
        t = torch.arange(max_len, dtype = torch.float)
        # Generating a 2D tensor fo frequencies using the inv_freq and t arrays (both 1D).
        # We use a special summation called einstein summation (Einsum) to form the 2D tensor here.
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        # Concatenate two of these frequency matrices along the last dimension (column - wise).
        emb = torch.cat((freqs, freqs), dim = -1)
        # Now, store their sines and cosines in the GPU registers as buffers.
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())

    # Now, we want to apply the transformation to the keys and queries.
    # This transformation will be require to apply to the all 64 dimensions of d_k.
    # We slice and concatenate the projected key and query matrices accordingly.
    def _rotate_half(self, x):
        x1 = x[..., :x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2:]
        # First few, last few concatenated.
        return torch.cat((-x2, x1), dim = -1)

    # We apply the transforms here in one forward pass, and then move on accordingly.
    def forward(self, tensor, seq_len):
        cos = self.cos_cached[:seq_len, :].unsqueeze(0).unsqueeze(1)
        sin = self.sin_cached[:seq_len, :].unsqueeze(0).unsqueeze(1)
        return (tensor * cos) + (self._rotate_half(tensor) * sin)

########################################
# 2.1.4 Relative Positional Encoding
########################################

# The classical relative positional encoding method (similar to Transformer-XL or T5).
# Instead of injecting absolute addresses, it trains a specific embedding vector for every possible
# distance between two tokens (e.g., "Token B is 5 steps ahead of Token A").
class RelativePositionalEncoding(nn.Module):
    def __init__(self, d_head, max_relative_position = 16):
        super().__init__()
        # We limit the tracking distance to a max threshold to save memory.
        # If a token is 100 steps away, it just gets the generic "16+ steps away" embedding.
        self.max_relative_position = max_relative_position
        # The vocab size needs to cover both negative (left) and positive (right) distances, plus 0.
        self.vocab_size = 2 * max_relative_position + 1
        # This is a standard learnable lookup table, mapping distance IDs to actual embedding vectors.
        self.embeddings_table = nn.Embedding(self.vocab_size, d_head)

    def forward(self, seq_len, device):
        # Generate an N x N matrix representing the exact distance between any two tokens in the window.
        range_vec = torch.arange(seq_len, device=device)
        distance_matrix = range_vec.unsqueeze(1) - range_vec.unsqueeze(0)
        # Clamp distances to our max threshold to prevent index-out-of-bounds errors on our embedding table.
        clamped_distances = torch.clamp(distance_matrix, -self.max_relative_position, self.max_relative_position)
        # Shift the values so the lowest possible distance (-16) becomes index 0.
        # nn.Embedding cannot process negative indices!
        lookup_indices = clamped_distances + self.max_relative_position
        return self.embeddings_table(lookup_indices)

In [ ]:
###################################################################
# 2.2.1 Attention Mechanisms: Vanilla Attention (Multiheaded)
###################################################################

class MultiHeadedAttention(nn.Module):
    def __init__(self, d_model, n_heads, pos_variant = "absolute"):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        self.pos_variant = pos_variant
        # d_k is the dimension of each individual head. (e.g. 256 / 8 = 32)
        self.d_k = d_model // n_heads

        # Linear projections to generate our Queries, Keys, and Values.
        self.W_key = nn.Linear(d_model, d_model)
        self.W_query = nn.Linear(d_model, d_model)
        self.W_value = nn.Linear(d_model, d_model)
        # Final projection to mix the output of all heads back together.
        self.fc_out = nn.Linear(d_model, d_model)

        # Initialize the specific positional tracking module selected in the config.
        if self.pos_variant == "alibi":
            self.alibi = AlibiPositionalBias(n_heads)
        elif self.pos_variant == "rope":
            self.rope = RotaryPositionalEmbedding(self.d_k)
        elif self.pos_variant == "relative":
            self.rel_pos = RelativePositionalEncoding(self.d_k)

    def forward(self, K, Q, V, mask = None):
        batch_size, seq_len = Q.size(0), Q.size(1)

        # 1. Project the inputs and reshape them into the "macro-tile" layout.
        # Shape goes from [Batch, Seq, d_model] -> [Batch, Seq, Heads, d_k] -> [Batch, Heads, Seq, d_k]
        # Transposing brings the sequence and head dimensions next to each other for easy 2D matmul.
        Query = self.W_query(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        Key = self.W_key(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        Value = self.W_value(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        # If using RoPE, we physically rotate the coordinate space of the Queries and Keys right here.
        if self.pos_variant == "rope":
            Query = self.rope(Query, seq_len)
            Key = self.rope(Key, seq_len)

        # 2. Calculate raw attention scores: Q * K^T
        # We divide by sqrt(d_k) to prevent the dot product values from growing too large and flattening the softmax gradients.
        Scores = torch.matmul(Query, Key.transpose(-2, -1)) / math.sqrt(self.d_k)

        # If using Relative or ALiBi, we inject the positional distance information directly into the raw scores here.
        if self.pos_variant == "relative":
            Scores = Scores + torch.einsum('bhqd, qkd->bhqk', Query, self.rel_pos(seq_len, Q.device))
        if self.pos_variant == "alibi":
            Scores = Scores + self.alibi(seq_len, Q.device)

        # 3. Apply the causal mask.
        # Any token position marked as '-inf' gets pushed to a probability of 0.0 in the softmax step.
        if mask is not None:
            Scores = Scores.masked_fill(mask == float('-inf'), float('-inf'))

        # 4. Softmax normalizes the scores so they all sum to 1 across the sequence.
        Attention = torch.softmax(Scores, dim=-1)

        # 5. Multiply the attention weights by the Values, and glue the multiple heads back together.
        # .transpose(1, 2) moves it back to [Batch, Seq, Heads, d_k]
        # .contiguous() is required because transpose fragments the memory layout in RAM.
        # .view() flattens the heads back into the unified [Batch, Seq, d_model] dimension.
        Emb_add = torch.matmul(Attention, Value).transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.fc_out(Emb_add)

############################################
# 2.2.2 Sliding window / Local attention
############################################

class LocalHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, window_size = 64, pos_variant = "absolute"):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model
        # Window size strictly bounds how far back in time a token is allowed to look.
        self.window_size = window_size
        self.pos_variant = pos_variant
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)

        if self.pos_variant == "alibi":
            self.alibi = AlibiPositionalBias(n_heads)
        elif self.pos_variant == "rope":
            self.rope = RotaryPositionalEmbedding(self.d_k)
        elif self.pos_variant == "relative":
            self.rel_pos = RelativePositionalEncoding(self.d_k)

    def forward(self, K, Q, V, mask = None):
        batch_size, seq_len = Q.size(0), Q.size(1)

        Query = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        Key = self.W_k(K).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        Value = self.W_v(V).view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        if self.pos_variant == "rope":
            Query = self.rope(Query, seq_len)
            Key = self.rope(Key, seq_len)

        Scores = torch.matmul(Query, Key.transpose(-2, -1)) / math.sqrt(self.d_k)

        if self.pos_variant == "relative":
            Scores = Scores + torch.einsum('bhqd, qkd->bhqk', Query, self.rel_pos(seq_len, Q.device))
        if self.pos_variant == "alibi":
            Scores = Scores + self.alibi(seq_len, Q.device)

        # Creates a lower triangular mask shifted down by 'window_size'.
        # This isolates a tight diagonal band. Anything outside the band gets zeroed out.
        if self.window_size < seq_len:
            past_mask = torch.tril(torch.ones(seq_len, seq_len, device = Q.device), diagonal = -self.window_size).bool()
            Scores = Scores.masked_fill(past_mask, float('-inf'))

        # Standard causal mask to block the future.
        if mask is not None:
            Scores = Scores.masked_fill(mask == float('-inf'), float('-inf'))

        Attention = torch.softmax(Scores, dim = -1)
        Emb_add = torch.matmul(Attention, Value).transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.fc_out(Emb_add)

###########################################
# 2.2.3 Kernelized attention
###########################################

class KernelAttention(nn.Module):
    def __init__(self, d_model, n_heads, seq_len, pos_variant = "absolute", chunk_size = 64):
        super().__init__()
        self.n_heads = n_heads
        self.d_model = d_model

        # Guardrails against invalid configurations.
        # Kernel attention bypasses Softmax, breaking the N x N distance map required by ALiBi/Relative.
        if pos_variant in ["alibi", "relative"]:
            raise ValueError(f"{pos_variant} requires an N x N matrix. Incompatible with Kernel Attention.")
        # RoPE geometrically shifts coordinates in ways that conflict with the ReLU transformation.
        if pos_variant == "rope":
            print("[WARNING] RoPE is mathematically incompatible with ReLU Kernel mapping. Defaulting to 'absolute'.")
        # self.pos_variant = "absolute"

        self.eps = 1e-6
        self.d_k = d_model // n_heads
        # Chunk size for parallel block execution to maximize GPU hardware cache efficiency.
        self.chunk_size = chunk_size

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, K, Q, V, mask = None):
        batch_size, seq_len = Q.size(0), Q.size(1)

        # Notice we are leaving the sequence dimension in the second slot: [Batch, Seq, Heads, d_k]
        # This layout makes slicing the sequence into chunks easier.
        Query = self.W_q(Q).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        Key = self.W_k(K).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        Value = self.W_v(V).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        # The core Kernel Trick: We drop Softmax and apply ReLU to ensure positivity.
        Query = torch.relu(Query) + self.eps
        Key = torch.relu(Key) + self.eps

        # These are our persistent memory banks. They hold the historical context of the entire sequence.
        # Because we multiply K^T and V first, this memory matrix never exceeds size (d_k x d_k).
        KV_state = torch.zeros(batch_size, self.n_heads, self.d_k, self.d_k, device = Q.device)
        K_state = torch.zeros(batch_size, self.n_heads, self.d_k, device = Q.device)

        # Mask strictly for the tokens inside the current 64-token block.
        chunk_mask = torch.tril(torch.ones(self.chunk_size, self.chunk_size, device = Q.device))
        outputs = []

        # Iterate over the sequence in chunks to balance GPU parallelism with RNN causality.
        for i in range(0, seq_len, self.chunk_size):
            # 1. Grab the current 64-token slice.
            q_chunk = Query[:, :, i:i+self.chunk_size, :]
            k_chunk = Key[:, :, i:i+self.chunk_size, :]
            v_chunk = Value[:, :, i:i+self.chunk_size, :]

            # Edge case handling for the final chunk if it doesn't divide perfectly by 64.
            actual_chunk_size = q_chunk.size(2)
            mask_chunk = chunk_mask[:actual_chunk_size, :actual_chunk_size]

            # 2. Local Interaction: Compute standard attention purely within this 64-token block.
            local_scores = torch.matmul(q_chunk, k_chunk.transpose(-2, -1))
            local_scores = local_scores * mask_chunk
            local_out = torch.matmul(local_scores, v_chunk)
            local_denom = local_scores.sum(dim = -1)

            # 3. Global Interaction: Multiply the Query chunk by the historical memory bank.
            # This instantly retrieves all the context from previous chunks in O(N) time.
            cross_out = torch.matmul(q_chunk, KV_state)
            cross_denom = torch.matmul(q_chunk, K_state.unsqueeze(-1)).squeeze(-1)

            # Combine local and global contexts and normalize.
            num = local_out + cross_out
            den = local_denom + cross_denom
            outputs.append(num / den.unsqueeze(-1))

            # 4. State Update: Compress the current chunk's K and V into a signature and add it to the bank.
            KV_state = KV_state + torch.matmul(k_chunk.transpose(-2, -1), v_chunk)
            K_state = K_state + k_chunk.sum(dim = 2)

        # Glue the processed chunks back into a continuous sequence tensor.
        Emb_add = torch.cat(outputs, dim = 2)
        Emb_add = Emb_add.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.fc_out(Emb_add)

##########################################
# 2.2.4 GQA (General Queried Attention)
##########################################

# General / Grouped Query Attention reduces GPU memory usage during generation by
# forcing multiple Query heads to share a single Key and Value head.
class GeneralAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_groups, pos_variant = "absolute"):
        super().__init__()
        self.n_heads = n_heads
        # n_groups defines how many distinct Key/Value clusters we will maintain.
        self.n_groups = n_groups
        self.d_model = d_model
        self.pos_variant = pos_variant
        self.d_head = d_model // n_heads
        self.heads_per_group = n_heads // n_groups

        self.W_q = nn.Linear(d_model, d_model)
        # Notice that W_k and W_v project to a much smaller dimensionality than W_q!
        # This saves massive amounts of VRAM since K and V caches dominate memory during inference.
        self.W_k = nn.Linear(d_model, n_groups * self.d_head)
        self.W_v = nn.Linear(d_model, n_groups * self.d_head)
        self.fc_out = nn.Linear(d_model, d_model)

        if self.pos_variant == "alibi":
            self.alibi = AlibiPositionalBias(n_heads)
        elif self.pos_variant == "rope":
            self.rope = RotaryPositionalEmbedding(self.d_head)
        elif self.pos_variant == "relative":
            self.rel_pos = RelativePositionalEncoding(self.d_head)

    def forward(self, K, Q, V, mask = None):
        batch_size, seq_len = Q.size(0), Q.size(1)

        # Query retains its full distinct 8 heads.
        Query = self.W_q(Q).view(batch_size, -1, self.n_heads, self.d_head).transpose(1, 2)
        # Keys and Values only generate 2 groups (acting as 2 fat heads).
        Key = self.W_k(K).view(batch_size, -1, self.n_groups, self.d_head).transpose(1, 2)
        Value = self.W_v(V).view(batch_size, -1, self.n_groups, self.d_head).transpose(1, 2)

        # The Zero-Stride Memory Trick:
        # We use .expand() to virtually duplicate the Key/Value groups to match the Query head count.
        # Because we use expand() instead of repeat(), PyTorch doesn't allocate any extra VRAM.
        # It just updates the tensor strides to read the same group multiple times.
        Key = Key.unsqueeze(2).expand(batch_size, self.n_groups, self.heads_per_group, seq_len, self.d_head)
        Key = Key.reshape(batch_size, self.n_heads, seq_len, self.d_head)

        Value = Value.unsqueeze(2).expand(batch_size, self.n_groups, self.heads_per_group, seq_len, self.d_head)
        Value = Value.reshape(batch_size, self.n_heads, seq_len, self.d_head)

        if self.pos_variant == "rope":
            Query = self.rope(Query, seq_len)
            Key = self.rope(Key, seq_len)

        Scores = torch.matmul(Query, Key.transpose(-2, -1)) / math.sqrt(self.d_head)

        if self.pos_variant == "relative":
            Scores = Scores + torch.einsum('bhqd, qkd->bhqk', Query, self.rel_pos(seq_len, Q.device))
        if self.pos_variant == "alibi":
            Scores = Scores + self.alibi(seq_len, Q.device)

        if mask is not None:
            Scores = Scores.masked_fill(mask == float('-inf'), float('-inf'))

        Attention = torch.softmax(Scores, dim=-1)
        x = torch.matmul(Attention, Value).transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.fc_out(x)

In [ ]:
#######################################
# 3. The Base transformer block
#######################################

# Represents a single repeated "layer" inside the overall transformer architecture.
class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout = 0.1, attn_variant = "vanilla", pos_variant = "absolute", n_groups = 2, window_size = 64, seq_len = 1024):
        super().__init__()

        # Dynamic selection of the attention mechanism based on configuration string.
        if attn_variant == "vanilla":
            self.attention = MultiHeadedAttention(d_model, n_heads, pos_variant = pos_variant)
        elif attn_variant == "local":
            self.attention = LocalHeadAttention(d_model, n_heads, window_size = window_size, pos_variant = pos_variant)
        elif attn_variant == "kernelized":
            self.attention = KernelAttention(d_model, n_heads, seq_len = seq_len, pos_variant = pos_variant)
        elif attn_variant == "gqa":
            self.attention = GeneralAttention(d_model, n_heads, n_groups = n_groups, pos_variant = pos_variant)
        else:
            raise ValueError(f"Unknown attention variant: {attn_variant}")

        # LayerNorm normalizes the vector distributions to stabilize deep gradients.
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        # Standard multi-layer perceptron (MLP) mapping to add non-linear transformations after attention.
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask = None):
        # We are using "Pre-LayerNorm" architecture (x = x + Sublayer(Norm(x))).
        # This allows the original untouched input 'x' to flow cleanly through the residual
        # connection, which prevents gradients from vanishing in very deep models.
        normed_x = self.norm1(x)
        attention_out = self.attention(normed_x, normed_x, normed_x, mask)
        x = x + self.dropout(attention_out)

        ff_out = self.feed_forward(self.norm2(x))
        x = x + self.dropout(ff_out)
        return x

# The overarching model wrapper that ties embeddings, encodings, and blocks together.
class Transformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, dropout = 0.1, attn_variant = "vanilla", pos_variant = "absolute", n_groups = 2, window_size = 64, seq_len = 1024):
        super().__init__()
        # Token embedding lookup table (maps token IDs to d_model space).
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_variant = pos_variant

        # Initialize the baseline additive encoding if required.
        if self.pos_variant == "absolute":
            self.pos_encoder = PositionalEncoder(d_model)
        else:
            # If using RoPE/ALiBi/Relative, positions are injected dynamically in the attention block.
            # nn.Identity() acts as a transparent passthrough to satisfy the forward loop structure.
            self.pos_encoder = nn.Identity()

        # Generate N sequential transformer layers.
        self.layers = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout, attn_variant, pos_variant, n_groups, window_size, seq_len) for _ in range(n_layers)
        ])

        # Pre-LN networks require a final normalization layer before generating logits.
        self.final_norm = nn.LayerNorm(d_model)
        # Final classifier layer pushing the d_model space back out to vocabulary probabilities.
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, x, d_model, mask = None):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        for layer in self.layers:
            x = layer(x, mask = mask)
        return self.fc_out(self.final_norm(x))

# Utility to create the triangular causal mask for autoregressive training.
def generate_square_subsequent_mask(sz):
    # triu creates an upper-triangular matrix, we transpose it to make it lower-triangular.
    mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
    # 0s (future tokens) become -inf (which softmax turns to 0).
    # 1s (past/present tokens) become 0.0 (which softmax ignores).
    mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
    return mask

In [ ]:
########################################
# 4. Training and Evaluation loops
########################################

# Hyperparameters setup
d_model = 512
n_heads = 8
n_layers = 6
d_ff = 2048
dropout = 0.1
lr = 0.0001
epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Loss function mapping the raw logit outputs against the target probability distribution.
criterion = nn.CrossEntropyLoss()

def train(model, train_data, val_data, optimizer, criterion, win_len, epoch, history):
    # .train() enables Dropouts and Normalization state tracking.
    model.train()
    start_time = time.time()
    tokens_processed = 0
    base_lr = 0.0001

    for batch, i in enumerate(range(0, train_data.size(0) - 1, win_len)):
        data, targets = get_batch(train_data, i, win_len)
        # Transpose repositions the data from [Seq, Batch] -> [Batch, Seq] for model ingestion.
        data = data.t().to(device)
        targets = targets.t().to(device)

        # Generate the causal diagonal mask for the current context window.
        mask = generate_square_subsequent_mask(data.size(1)).to(device)

        # Learning Rate Linear Warmup: Ramp up the LR over the first 100 steps.
        # This prevents the raw, uninitialized gradients from destabilizing the network early on.
        global_step = (epoch - 1) * (train_data.size(0) // win_len) + batch
        if global_step < 100:
            lr_scale = float(global_step) / 100.0
            for param_group in optimizer.param_groups:
                param_group['lr'] = base_lr * lr_scale

        # Reset gradients before the backward pass.
        optimizer.zero_grad()
        output = model(data, d_model, mask = mask)

        # View(-1) collapses the [Batch, Seq] grid into a flat 1D sequence to match the targets.
        loss = criterion(output.view(-1, vocab_size), targets.reshape(-1))
        loss.backward()

        # Gradient clipping physically caps the size of gradient steps at 0.5.
        # Exploding gradients are common in deep transformers, this prevents weights from shooting to infinity.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()

        tokens_processed += data.numel()

        # Logging logic for console and plot generation.
        if batch % 10 == 0 and batch > 0:
            elapsed = time.time() - start_time
            throughput = tokens_processed / elapsed
            current_lr = optimizer.param_groups[0]['lr']

            if torch.cuda.is_available():
                gpu_max_allocated = torch.cuda.max_memory_allocated()
                gpu_total = torch.cuda.get_device_properties(device).total_memory
                gpu_mem_pct = (gpu_max_allocated / gpu_total) * 100
            else:
                gpu_mem_pct = 0.0

            # Perplexity is mathematically e^(Loss).
            try:
                train_ppl = math.exp(loss.item())
            except OverflowError:
                train_ppl = float('inf')

            print(f"Epoch {epoch} | Batch {batch:4d} | LR {current_lr:.2e} | Train Loss {loss.item():.4f} | Train PPL {train_ppl:.2f} | Throughput {throughput:.1f} tok/s")

            # Update historical tracking dictionary.
            current_step = global_step
            history['step'].append(current_step)
            history['train_loss'].append(loss.item())
            history['gpu_pct'].append(gpu_mem_pct)
            history['throughput'].append(throughput)

            # Pad validation lists so the graphing axes align cleanly later.
            history['val_loss'].append(history['val_loss'][-1] if history['val_loss'] else loss.item())
            history['ppl'].append(history['ppl'][-1] if history['ppl'] else math.exp(loss.item()))

            start_time = time.time()
            tokens_processed = 0

def evaluate(model, data_source, win_len):
    # .eval() disables dropouts for a clean, deterministic inference pass.
    model.eval()
    total_loss = 0.
    # torch.no_grad() disables the computational memory graph saving us massive amounts of VRAM.
    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, win_len):
            data, targets = get_batch(data_source, i, win_len)
            data = data.t().to(device)
            targets = targets.t().to(device)
            mask = generate_square_subsequent_mask(data.size(1)).to(device)
            output = model(data, d_model, mask = mask)
            loss = criterion(output.view(-1, vocab_size), targets.reshape(-1))
            total_loss += data.size(1) * loss.item()
    return total_loss / (len(data_source) - 1)

def benchmark_inference(model, prompt_len = 64, gen_len = 64, batch_size = 8):
    model.eval()
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    # Generate a dummy input block of token IDs to act as a simulated prompt.
    input_ids = torch.randint(1, vocab_size, (batch_size, prompt_len)).to(device)
    start_time = time.time()

    with torch.no_grad():
        # Autoregressive generation loop: Generates tokens one at a time sequentially.
        for _ in range(gen_len):
            mask = generate_square_subsequent_mask(input_ids.size(1)).to(device)
            output = model(input_ids, d_model, mask = mask)
            # Extract only the probabilities corresponding to the absolute last generated token.
            next_token_logits = output[:, -1, :]
            # Greedy decoding: Selects the absolute highest probability word as the next token.
            next_token = torch.argmax(next_token_logits, dim = -1, keepdim = True)
            # Paste the generated token onto the context window to feed back into the next loop iteration.
            input_ids = torch.cat([input_ids, next_token], dim = 1)

    end_time = time.time()
    total_tokens_generated = batch_size * gen_len
    throughput = total_tokens_generated / (end_time - start_time)
    peak_mem = torch.cuda.max_memory_allocated() / (1024 ** 2) if torch.cuda.is_available() else 0

    return throughput, peak_mem

# Function to physically compress the model weights for down-stream GQA inference.
def convert_mha_to_gqa(input_ckpt, output_ckpt, n_heads, n_groups, d_model):
    old_state_dict = torch.load(input_ckpt, map_location = 'cpu')
    new_state_dict = {}
    d_k = d_model // n_heads
    heads_per_group = n_heads // n_groups

    for key, tensor in old_state_dict.items():
        # Isolate Key and Value projection layers.
        if 'attention.W_key' in key or 'attention.W_value' in key:
            new_key = key.replace('W_key', 'W_k').replace('W_value', 'W_v')
            if 'weight' in key:
                reshaped = tensor.view(n_groups, heads_per_group, d_k, d_model)
                # Mean pool the head weights within the group to compress memory overhead.
                new_tensor = reshaped.mean(dim = 1).view(n_groups * d_k, d_model)
            elif 'bias' in key:
                reshaped = tensor.view(n_groups, heads_per_group, d_k)
                new_tensor = reshaped.mean(dim = 1).view(-1)
            new_state_dict[new_key] = new_tensor
        elif 'attention.W_query' in key:
            # Query is untouched so it retains its granular resolution mapping.
            new_key = key.replace('W_query', 'W_q')
            new_state_dict[new_key] = tensor
        else:
            new_state_dict[key] = tensor
    torch.save(new_state_dict, output_ckpt)

#################################################################
# 5. Plot generation
#################################################################

def plot_training_dynamics(history, filename = "model_variant"):
    print(f"\nGenerating Training Dynamics Plots for {filename}...")
    sns.set_theme(style = "whitegrid")

    steps = history['step']
    fig, axs = plt.subplots(2, 2, figsize = (14, 10))

    # 1. Log-Loss Tracking over Batches
    axs[0, 0].plot(steps, history['train_loss'], label = 'Train Loss', alpha = 0.8)
    axs[0, 0].plot(steps, history['val_loss'], label = 'Val Loss', alpha = 0.8, linestyle = '--')
    axs[0, 0].set_yscale('log')
    axs[0, 0].set_title('Log-Loss over Time')
    axs[0, 0].set_xlabel('Steps (Batches)')
    axs[0, 0].set_ylabel('Log Loss')
    axs[0, 0].legend()

    # 2. Perplexity (True language modeling confidence metric)
    axs[0, 1].plot(steps, history['ppl'], color = 'purple')
    axs[0, 1].set_title('Validation Perplexity over Time')
    axs[0, 1].set_xlabel('Steps (Batches)')
    axs[0, 1].set_ylabel('PPL')

    # 3. Peak VRAM monitoring over time
    axs[1, 0].plot(steps, history['gpu_pct'], color = 'red')
    axs[1, 0].set_title('Peak GPU VRAM Usage % over Time')
    axs[1, 0].set_xlabel('Steps (Batches)')
    axs[1, 0].set_ylabel('GPU Memory Allocation (%)')

    # 4. Token processing speed (useful for verifying memory bandwidth bottlenecks)
    axs[1, 1].plot(steps, history['throughput'], color = 'green')
    axs[1, 1].set_title('Training Throughput over Time')
    axs[1, 1].set_xlabel('Steps (Batches)')
    axs[1, 1].set_ylabel('Tokens / second')

    plt.tight_layout()
    plt.savefig(f"{filename}.pdf")
    plt.close()

##########################################################################
# 6. Single Run Configuration Pipeline
##########################################################################

if __name__ == "__main__":
    import gc
    import os
    import math # Ensure math is imported for math.exp
    import time

    # Set hyperparameters for the specific Attention/Positional variants desired.
    SELECTED_ATTN = "gqa"
    SELECTED_POS  = "relative"

    # We removed the RoPE guardrail as requested for the experiment!
    # However, we must keep the guardrail for ALiBi/Relative, as Kernelized
    # attention physically does not generate the N x N grid required to apply them.
    if SELECTED_ATTN == "kernelized" and SELECTED_POS in ["alibi", "relative"]:
        raise ValueError(f"\n[GUARDRAIL TRIGGERED]: Incompatible configurations. Kernelized cannot process N x N distances.")

    n_groups = 2
    history = {'step': [], 'train_loss': [], 'val_loss': [], 'ppl': [], 'gpu_pct': [], 'throughput': []}

    print(f"\n{'='*70}")
    print(f"STARTING SINGLE EXPERIMENT: Attention = {SELECTED_ATTN.upper()} | Position = {SELECTED_POS.upper()}")
    print(f"{'='*70}\n")

    # For GQA, we start the baseline training cycle using standard Multi-Headed structure first.
    base_attn = "vanilla" if SELECTED_ATTN == "gqa" else SELECTED_ATTN

    try:
        model = Transformer(vocab_size, d_model, n_layers, n_heads, d_ff, dropout,
                            attn_variant = base_attn, pos_variant = SELECTED_POS).to(device)

        # Weight Tying
        model.fc_out.weight = model.embedding.weight

        # Xavier Uniform Baseline Initialization
        for p in model.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

        optimizer = torch.optim.Adam(model.parameters(), lr = lr)

        for epoch in range(1, epochs + 1):
            epoch_start = time.time()

            train(model, train_data, val_data, optimizer, criterion, win_len = 512, epoch = epoch, history = history)

            epoch_time = time.time() - epoch_start
            val_loss = evaluate(model, val_data, win_len = 512)
            ppl = math.exp(val_loss)

            if history['val_loss']:
                history['val_loss'][-1] = val_loss
                history['ppl'][-1] = ppl

            print(f"\n--> Epoch {epoch} completed in {epoch_time:.2f}s | Final Val Loss: {val_loss:.4f} | Final PPL: {ppl:.2f}\n")

        # The GQA secondary processing block.
        if SELECTED_ATTN == "gqa":
            print(f"\n{'='*60}\nConverting MHA to GQA and Fine-tuning\n{'='*60}")
            torch.save(model.state_dict(), 'tmp_pre_gqa.pt')
            convert_mha_to_gqa('tmp_pre_gqa.pt', 'tmp_gqa.pt', n_heads, n_groups, d_model)

            model = Transformer(vocab_size, d_model, n_layers, n_heads, d_ff, dropout,
                                attn_variant = "gqa", pos_variant = SELECTED_POS, n_groups = n_groups).to(device)
            model.load_state_dict(torch.load('tmp_gqa.pt', map_location=device))

            optimizer_gqa = torch.optim.Adam(model.parameters(), lr = 0.00005)

            epoch_start = time.time()
            train(model, train_data, val_data, optimizer_gqa, criterion, win_len = 512, epoch = epochs + 1, history = history)

            val_loss = evaluate(model, val_data, win_len = 512)
            ppl = math.exp(val_loss)
            if history['val_loss']:
                history['val_loss'][-1] = val_loss
                history['ppl'][-1] = ppl

            epoch_time = time.time() - epoch_start
            print(f"--> GQA Fine-tuning Epoch completed in {epoch_time:.2f}s | Final GQA PPL: {ppl:.2f}\n")

        print(f"\n{'='*60}\nTRAINING COMPLETE. EXECUTING EXTRAPOLATION TESTS\n{'='*60}")

        # --- NEW EXTRAPOLATION SWEEP BLOCK ---
        extrapolation_lengths = [512, 1024, 2048]
        for wl in extrapolation_lengths:
            try:
                # We disable gradient tracking inside evaluate() anyway, so this is purely an inference pass
                extrap_loss = evaluate(model, val_data, win_len = wl)
                extrap_ppl = math.exp(extrap_loss)
                print(f"[EXTRAPOLATION] Context Length: {wl:<4} | Validation PPL: {extrap_ppl:.2f}")
            except Exception as e:
                # This will likely trigger for 'absolute' encodings at 1024/2048
                print(f"[EXTRAPOLATION] Context Length: {wl:<4} | FAILED! (Error: {type(e).__name__})")
        # -------------------------------------

        print(f"\n{'='*60}\nGENERATING METRICS & PLOTS\n{'='*60}")
        file_name_string = f"{SELECTED_ATTN}_{SELECTED_POS}"
        plot_training_dynamics(history, filename=file_name_string)

        inf_throughput, inf_mem = benchmark_inference(model)
        print(f"Post-Train Inference Speed: {inf_throughput:.1f} tok/s | Peak Mem: {inf_mem:.1f}MB")

    except Exception as e:
        print(f"\n[ERROR]: The configuration crashed during execution: {str(e)}")

    finally:
        # The critical VRAM purge to ensure you can run the next combo safely
        if 'model' in locals(): del model
        if 'optimizer' in locals(): del optimizer
        if 'optimizer_gqa' in locals(): del optimizer_gqa
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Colab specific integration to force download
    if IN_COLAB:
        try:
            from google.colab import files
            print("\nPushing files to local browser download...")
            if os.path.exists(f"{file_name_string}.pdf"):
                files.download(f"{file_name_string}.pdf")
        except Exception as e:
            pass


STARTING SINGLE EXPERIMENT: Attention = GQA | Position = RELATIVE

Epoch 1 | Batch   10 | LR 1.00e-05 | Train Loss 10.7510 | Train PPL 46678.35 | Throughput 9739.3 tok/s
Epoch 1 | Batch   20 | LR 2.00e-05 | Train Loss 10.5966 | Train PPL 39997.68 | Throughput 10583.4 tok/s
Epoch 1 | Batch   30 | LR 3.00e-05 | Train Loss 10.4204 | Train PPL 33535.72 | Throughput 10504.4 tok/s
Epoch 1 | Batch   40 | LR 4.00e-05 | Train Loss 10.2483 | Train PPL 28233.95 | Throughput 10427.8 tok/s
Epoch 1 | Batch   50 | LR 5.00e-05 | Train Loss 9.9902 | Train PPL 21812.54 | Throughput 10384.3 tok/s
Epoch 1 | Batch   60 | LR 6.00e-05 | Train Loss 9.7582 | Train PPL 17295.93 | Throughput 10418.7 tok/s
Epoch 1 | Batch   70 | LR 7.00e-05 | Train Loss 9.4413 | Train PPL 12598.16 | Throughput 10332.5 tok/s
Epoch 1 | Batch   80 | LR 8.00e-05 | Train Loss 9.0923 | Train PPL 8886.68 | Throughput 10353.5 tok/s
Epoch 1 | Batch   90 | LR 9.00e-05 | Train Loss 8.7802 | Train PPL 6504.17 | Throughput 10319.8 tok/s
Epoc

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>